# Ablation study

Ablations of the paper's CellBridge model, the four-layer, all-training-cells, adapted representation. HEST excludes TENX143.

1. **Teacher ablation.** Macro F1 of CellBridge against NOVAE, the same model trained with the NOVAE teacher (also four LoRA layers, all training cells, adapted representation), on every benchmark with a linear-probe classifier.
2. **CovGap ablation.** Class-conditional coverage gap of CellBridge under a chosen set of conformal constructions. Set `COVGAP_METHODS` in the configuration cell to pick which constructions to keep, in the order they should appear.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/cellbridge").is_dir())
WORKSPACE = Path(os.environ.get("CELLBRIDGE_WORKSPACE", REPO.parents[1]))
OUTPUTS_ROOT = Path(os.environ.get("CELLBRIDGE_OUTPUTS_ROOT", WORKSPACE / "outputs"))
CELLBRIDGE_RESULTS_ROOT = Path(os.environ.get(
    "CELLBRIDGE_4L_INFERENCE_RESULTS",
    OUTPUTS_ROOT / "reproducibility/inference/cellbridge/uni_terra_four_layers_adapted_all_cells",
))
NOVAE_RESULTS_ROOT = Path(os.environ.get(
    "NOVAE_4L_INFERENCE_RESULTS",
    OUTPUTS_ROOT / "reproducibility/inference/cellbridge/uni_novae_four_layers_adapted_all_cells",
))

# Model label -> inference root. Both runs are four-layer, all-cells.
TEACHER_RUNS = {
    "NOVAE": NOVAE_RESULTS_ROOT,
    "CellBridge": CELLBRIDGE_RESULTS_ROOT,
}
OURS = "CellBridge"
ARM = "lora_adapted"
PROBES = ["linear", "knn"]
PROBE_LABELS = {"linear": "Linear probe", "knn": "kNN"}
EXCLUDED_HEST_SAMPLES = {"TENX143"}

# (dataset, task, table label, LaTeX label) in table order; HEST, the only ID set, first.
BENCHMARKS = [
    ("hest", "all", "HEST", "HEST"),
    ("nucls", "all", "NuCLS", "NuCLS"),
    ("pannuke", "all", "PanNuke", "PanNuke"),
    ("lizard", "all", "Lizard", "Lizard"),
    ("phenocell", "all", "PhenoCell all", r"PhenoCell\textsubscript{all}"),
    ("phenocell", "immune", "PhenoCell immune", r"PhenoCell\textsubscript{imm}"),
]

# CovGap ablation: keep any subset of CONFORMAL_LABELS, in column order.
COVGAP_METHODS = ["split", "graph", "branch_graph"]
COVGAP_PROBE = "linear"
CONFORMAL_LABELS = {
    "split": "Standard",
    "graph": "Graph",
    "class_split": "Class split",
    "class_graph": "Class graph",
    "branch_split": "Branch split",
    "branch_graph": "Ours",
    "hybrid_split": "Hybrid split",
    "hybrid_graph": "Hybrid graph",
}
# LaTeX column header for each construction.
CONFORMAL_LATEX_LABELS = {
    "split": "Standard",
    "graph": "Graph",
    "class_split": "Class split",
    "class_graph": "Class graph",
    "branch_split": "Branch split",
    "branch_graph": "Ours",
    "hybrid_split": "Hybrid split",
    "hybrid_graph": "Hybrid graph",
}


## Load results

Both runs keep dataset directories directly below their inference root, with HEST split into one directory per section. Loading fails if a model is missing a benchmark or probe, or if the runs cover different HEST sections.


In [ ]:
BENCHMARK_BY_KEY = {(dataset, task): label for dataset, task, label, _ in BENCHMARKS}
BENCHMARK_LABELS = [label for _, _, label, _ in BENCHMARKS]
N_OOD = sum(dataset != "hest" for dataset, *_ in BENCHMARKS)


def iter_run_dirs(root):
    """Yield (dataset, sample_id, run_dir); HEST has one run per section."""
    if not root.is_dir():
        raise FileNotFoundError(f"Results root does not exist: {root}")
    for dataset_dir in sorted(path for path in root.iterdir() if path.is_dir()):
        if dataset_dir.name != "hest":
            yield dataset_dir.name, None, dataset_dir
            continue
        for sample_dir in sorted(path for path in dataset_dir.iterdir() if path.is_dir()):
            if sample_dir.name not in EXCLUDED_HEST_SAMPLES:
                yield "hest", sample_dir.name, sample_dir


def load_macro_f1(runs):
    rows = []
    for model, root in runs.items():
        for dataset, sample_id, run_dir in iter_run_dirs(root):
            for path in sorted(run_dir.glob(f"evaluation/{ARM}/*/*/metrics.csv")):
                task, probe = path.parent.parent.name, path.parent.name
                benchmark = BENCHMARK_BY_KEY.get((dataset, task))
                if benchmark is None or probe not in PROBES:
                    continue
                metrics = pd.read_csv(path)
                if len(metrics) != 1:
                    raise ValueError(f"Expected one row in {path}, found {len(metrics)}")
                bootstrap_se = np.nan
                uncertainty_path = path.with_name("uncertainty.csv")
                if uncertainty_path.exists():
                    errors = pd.read_csv(uncertainty_path).set_index("metric")
                    bootstrap_se = errors.loc["macro_f1", "bootstrap_se"]
                rows.append({
                    "model": model, "probe": probe, "benchmark": benchmark,
                    "sample_id": sample_id, "macro_f1": metrics["macro_f1"].iloc[0],
                    "bootstrap_se": bootstrap_se,
                })
    return pd.DataFrame(rows)


def summarise_macro_f1(scores):
    """One value and standard error per (model, probe, benchmark)."""
    summary = []
    for (model, probe, benchmark), group in scores.groupby(["model", "probe", "benchmark"]):
        if benchmark == "HEST":
            # HEST: SE across sections; OOD: saved grouped-bootstrap SE.
            value, error = group["macro_f1"].mean(), group["macro_f1"].sem(ddof=1)
        elif len(group) == 1:
            value, error = group["macro_f1"].iloc[0], group["bootstrap_se"].iloc[0]
        else:
            raise ValueError(
                f"Expected one result for {(model, probe, benchmark)}, found {len(group)}"
            )
        summary.append({
            "model": model, "probe": probe, "benchmark": benchmark,
            "macro_f1": value, "error": error,
        })
    return pd.DataFrame(summary)


def load_covgap(root, probe):
    """Per-run CovGap rows for every conformal construction."""
    tables = []
    for dataset, sample_id, run_dir in iter_run_dirs(root):
        paths = run_dir.glob(f"evaluation/{ARM}/*/{probe}/conformal_prediction/metrics.csv")
        for path in sorted(paths):
            benchmark = BENCHMARK_BY_KEY.get((dataset, path.parents[2].name))
            if benchmark is None:
                continue
            table = pd.read_csv(
                path, usecols=["conformal_method", "target_coverage", "coverage_gap"]
            )
            table["bootstrap_se"] = np.nan
            if dataset != "hest":
                uncertainty_path = path.with_name("uncertainty.csv")
                uncertainty = pd.read_csv(uncertainty_path)
                uncertainty = uncertainty.loc[uncertainty["metric"] == "coverage_gap"].set_index("conformal_method")
                stored = uncertainty.loc[table["conformal_method"]]
                if not np.allclose(stored["estimate"], table["coverage_gap"]):
                    raise ValueError(f"{uncertainty_path} disagrees with {path}")
                table["bootstrap_se"] = stored["bootstrap_se"].to_numpy()
            table["benchmark"] = benchmark
            table["sample_id"] = sample_id
            tables.append(table)
    if not tables:
        raise FileNotFoundError(f"No conformal metrics found below {root}")
    return pd.concat(tables, ignore_index=True)


scores = load_macro_f1(TEACHER_RUNS)
hest_sections = {
    key: frozenset(group["sample_id"])
    for key, group in scores.loc[scores["benchmark"] == "HEST"].groupby(["model", "probe"])
}
if len(set(hest_sections.values())) != 1:
    raise ValueError(f"The runs cover different HEST sections: {hest_sections}")
N_HEST_SECTIONS = len(next(iter(hest_sections.values())))

macro_f1 = summarise_macro_f1(scores)
expected = {
    (model, probe, label)
    for model in TEACHER_RUNS
    for probe in PROBES
    for label in BENCHMARK_LABELS
}
observed = set(macro_f1[["model", "probe", "benchmark"]].itertuples(index=False, name=None))
if expected - observed:
    raise ValueError(f"Missing macro F1 results: {sorted(expected - observed)}")

covgap_runs = load_covgap(CELLBRIDGE_RESULTS_ROOT, COVGAP_PROBE)
target_coverages = covgap_runs["target_coverage"].unique()
if len(target_coverages) != 1:
    raise ValueError(f"Expected one nominal coverage target, found {sorted(target_coverages)}")
NOMINAL_COVERAGE = float(target_coverages[0])
COVGAP_HEST_SECTIONS = covgap_runs.loc[covgap_runs["benchmark"] == "HEST", "sample_id"].nunique()
covgap = (
    covgap_runs.groupby(["conformal_method", "benchmark"], as_index=False).agg(
        coverage_gap=("coverage_gap", "mean"),
        error=("coverage_gap", "sem"),
        bootstrap_se=("bootstrap_se", "first"),
    )
)
ood = covgap["benchmark"] != "HEST"
covgap.loc[ood, "error"] = covgap.loc[ood, "bootstrap_se"]
covgap = covgap.drop(columns="bootstrap_se")

print(
    f"Macro F1: {', '.join(TEACHER_RUNS)} on {len(BENCHMARKS)} benchmarks "
    f"x {len(PROBES)} probes; HEST uses {N_HEST_SECTIONS} sections."
)
print(
    f"CovGap: {covgap['conformal_method'].nunique()} conformal constructions at "
    f"{NOMINAL_COVERAGE:.0%} nominal coverage from {CELLBRIDGE_RESULTS_ROOT}"
)

## Teacher ablation: macro F1

CellBridge against NOVAE, the same model trained with the NOVAE teacher. Both use four LoRA layers, all training cells, and the adapted representation, and are evaluated on every benchmark with a linear-probe classifier. HEST reports the mean ± standard error over the retained sections (sample standard deviation divided by the square root of the section count); the OOD benchmarks use their fixed test splits and saved grouped-bootstrap standard errors.

Datasets form the rows, grouped into ID and OOD; the columns compare the NOVAE and TERRA (CellBridge) teachers. The table shows linear-probe macro F1 only. The Difference column is TERRA minus NOVAE in percentage points, calculated from the displayed one-decimal scores.


In [ ]:
def _percentage(value, error=np.nan):
    r"""One cell as ``XX.X \scriptsize{$\pm$ X.XX}`` on a 0-100 scale."""
    if pd.isna(value):
        return "--"
    text = f"{100 * value:.1f}"
    if pd.notna(error):
        text = f"{text} \\scriptsize{{$\\pm$ {100 * error:.2f}}}"
    return text


def _plain(value, error=np.nan):
    if pd.isna(value):
        return "--"
    text = f"{100 * value:.1f}"
    return f"{text} ± {100 * error:.2f}" if pd.notna(error) else text


lookup = macro_f1.loc[macro_f1["probe"] == "linear"].set_index(["model", "benchmark"])
teacher_labels = {"CellBridge": "TERRA", "NOVAE": "NOVAE"}

teacher_table = pd.DataFrame(
    [
        [_plain(*lookup.loc[(model, benchmark), ["macro_f1", "error"]]) for model in TEACHER_RUNS]
        for benchmark in BENCHMARK_LABELS
    ],
    index=pd.MultiIndex.from_tuples(
        [("ID" if dataset == "hest" else "OOD", label) for dataset, _, label, _ in BENCHMARKS],
        names=["Split", "Dataset"],
    ),
    columns=pd.Index([teacher_labels[model] for model in TEACHER_RUNS], name="Teacher"),
)


# Difference in percentage points, calculated from the displayed scores.
teacher_difference = {
    benchmark: f"{round(100 * lookup.loc[('CellBridge', benchmark), 'macro_f1'], 1) - round(100 * lookup.loc[('NOVAE', benchmark), 'macro_f1'], 1):+.1f}"
    for benchmark in BENCHMARK_LABELS
}
teacher_table["Difference"] = [teacher_difference[benchmark] for benchmark in BENCHMARK_LABELS]
display(
    teacher_table.style.set_properties(**{"text-align": "right"})
    .set_properties(subset=["Difference"], **{"text-align": "center"})
)

lines = [
    r"\begin{wraptable}{r}{0.55\textwidth}",
    r"\centering",
    r"\footnotesize",
    r"\setlength{\tabcolsep}{4pt}",
    r"\vspace{-10pt}",
    r"\caption{Teacher ablation.}",
    r"\label{tab:ablation-teacher}",
    r"\begin{tabular}{llrrc}",
    r"\toprule",
    r" & \textbf{Dataset} & "
    + " & ".join(r"\textbf{" + teacher_labels[model] + "}" for model in TEACHER_RUNS)
    + r" & \textbf{Difference} \\",
    r"\midrule",
]
for i, (dataset, _, benchmark, latex) in enumerate(BENCHMARKS):
    if dataset == "hest":
        split = r"\rotatebox[origin=c]{90}{\textbf{ID}}"
    elif i == 1:
        lines.append(r"\midrule")
        split = r"\multirow{" + str(N_OOD) + r"}{*}{\rotatebox[origin=c]{90}{\textbf{OOD}}}"
    else:
        split = ""
    cells = []
    for model in TEACHER_RUNS:
        text = _percentage(*lookup.loc[(model, benchmark), ["macro_f1", "error"]])
        cells.append(text)
    cells.append(teacher_difference[benchmark])
    lines.append(split + " & " + latex + " & " + " & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}", r"\vspace{-5pt}", r"\end{wraptable}"]
print("\n".join(lines))


## CovGap ablation

Class-conditional coverage gap (CovGap; Ding et al., 2023) of CellBridge under the conformal constructions listed in `COVGAP_METHODS`: the mean absolute deviation of per-class coverage from the nominal target, in percentage points, averaged over the classes present in each test set. Lower is better, and 0 means every class is covered at exactly the target. Because the deviation is absolute, over-coverage of a class counts as much as under-coverage. HEST entries report the mean ± SE over the retained sections (sample standard deviation divided by the square root of the section count). OOD entries use the grouped-bootstrap SE from `conformal_prediction/uncertainty.csv`, with calibration thresholds held fixed, matching notebook 08. Both values and errors are in percentage points; errors have two decimal places.

Methods are columns, in the order listed in `COVGAP_METHODS`; datasets are rows, grouped into ID and OOD. By default the columns are split conformal prediction (Vovk et al., 2005), global graph conformal prediction (the closest analogue of Corbetta et al.), and ours, branch-conditional graph conformal prediction; `CONFORMAL_LATEX_LABELS` sets the LaTeX headers. Hybrid calibration never falls back to branch thresholds in this run, so hybrid columns repeat the class columns.


In [ ]:
unknown = [method for method in COVGAP_METHODS if method not in CONFORMAL_LABELS]
if unknown:
    raise ValueError(f"Unknown conformal methods {unknown}; choose from {list(CONFORMAL_LABELS)}")
if len(set(COVGAP_METHODS)) != len(COVGAP_METHODS):
    raise ValueError("COVGAP_METHODS lists a method more than once")

covgap_grid = (
    covgap.loc[covgap["conformal_method"].isin(COVGAP_METHODS)]
    .pivot(index="benchmark", columns="conformal_method", values="coverage_gap")
    .reindex(index=BENCHMARK_LABELS, columns=COVGAP_METHODS)
)
if covgap_grid.isna().any().any():
    raise ValueError("Missing CovGap entries:\n" + covgap_grid.isna().to_string())
covgap_errors = (
    covgap.pivot(index="benchmark", columns="conformal_method", values="error")
    .reindex(index=BENCHMARK_LABELS, columns=COVGAP_METHODS)
)
if covgap_errors.isna().any().any():
    raise ValueError("Missing CovGap standard errors:\n" + covgap_errors.isna().to_string())
covgap_shown = covgap_grid.map(lambda value: f"{value:.1f}")
covgap_rank = covgap_shown.astype(float).rank(axis=1, method="dense")

covgap_display = covgap_shown + " ± " + covgap_errors.map(lambda error: f"{error:.2f}")
covgap_display.index = pd.MultiIndex.from_tuples(
    [("ID" if dataset == "hest" else "OOD", label) for dataset, _, label, _ in BENCHMARKS],
    names=["Split", "Dataset"],
)
covgap_display.columns = pd.Index(
    [CONFORMAL_LABELS[method] for method in COVGAP_METHODS], name="Method",
)


covgap_styles = covgap_rank.map(
    lambda rank: "font-weight: bold" if rank == 1 else "text-decoration: underline" if rank == 2 else ""
)
covgap_styles.index = covgap_display.index
covgap_styles.columns = covgap_display.columns
display(
    covgap_display.style.apply(lambda _: covgap_styles, axis=None)
    .set_properties(**{"text-align": "right"})
)

lines = [
    r"\begin{wraptable}{r}{0.55\textwidth}",
    r"\centering",
    r"\footnotesize",
    r"\setlength{\tabcolsep}{4pt}",
    r"\vspace{-10pt}",
    r"\caption{CovGap ablation.}",
    r"\label{tab:ablation-covgap}",
    r"\begin{tabular}{ll" + "r" * len(COVGAP_METHODS) + "}",
    r"\toprule",
    r" & \textbf{Dataset} & "
    + " & ".join(r"\textbf{" + CONFORMAL_LATEX_LABELS[method] + "}" for method in COVGAP_METHODS)
    + r" \\",
    r"\midrule",
]
for i, (dataset, _, benchmark, latex) in enumerate(BENCHMARKS):
    if dataset == "hest":
        split = r"\rotatebox[origin=c]{90}{\textbf{ID}}"
    elif i == 1:
        lines.append(r"\midrule")
        split = r"\multirow{" + str(N_OOD) + r"}{*}{\rotatebox[origin=c]{90}{\textbf{OOD}}}"
    else:
        split = ""
    cells = []
    for method in COVGAP_METHODS:
        value = covgap_shown.loc[benchmark, method]
        error = covgap_errors.loc[benchmark, method]
        text = f"{value} \\scriptsize{{$\\pm$ {error:.2f}}}"
        rank = covgap_rank.loc[benchmark, method]
        if rank == 1:
            text = r"\textbf{" + text + "}"
        elif rank == 2:
            text = r"\underline{" + text + "}"
        cells.append(text)
    lines.append(split + " & " + latex + " & " + " & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}", r"\vspace{-5pt}", r"\end{wraptable}"]
print("\n".join(lines))


## Reading the tables

- Macro F1 values are percentages. HEST reports the mean ± standard error across the retained sections; the OOD benchmarks report grouped-bootstrap standard errors.
- CovGap errors follow notebook 08: SE over HEST sections and saved grouped-bootstrap SE for OOD datasets.
- In CovGap, the lowest displayed value per dataset is bold and the second-lowest is underlined, among the selected methods. Ties receive the same emphasis.
